### 1. Load configuration
Load catalog names, schemas, and the source table list.

In [0]:
%run ./00_config

### 2. Profile Bronze: inspect column types
Print the schema of every Bronze table to decide which data types to use in the Silver DDL.

In [0]:
for t in TABLES:
    print(f"\n=== {t} ===")
    spark.table(f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.{t}").printSchema()

### 3. Profile Bronze: key uniqueness
Check that every primary key is unique and non-null in Bronze before declaring it in Silver.

In [0]:
keys = {
    "region":   ["r_regionkey"],
    "nation":   ["n_nationkey"],
    "supplier": ["s_suppkey"],
    "customer": ["c_custkey"],
    "part":     ["p_partkey"],
    "partsupp": ["ps_partkey", "ps_suppkey"],
    "orders":   ["o_orderkey"],
    "lineitem": ["l_orderkey", "l_linenumber"],
}

rows = []
for t, cols in keys.items():
    df = spark.table(f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.{t}")
    total = df.count()
    distinct = df.select(*cols).distinct().count()
    nulls = df.filter(" OR ".join(f"{c} IS NULL" for c in cols)).count()
    rows.append((t, ", ".join(cols), total, distinct, nulls, total == distinct and nulls == 0))

display(spark.createDataFrame(
    rows,
    "table_name string, key_columns string, total_rows long, distinct_keys long, null_keys long, key_is_valid boolean",
))

invalid_keys = [
    (table, key_columns, total, distinct, nulls)
    for table, key_columns, total, distinct, nulls, valid in rows
    if not valid
]

assert not invalid_keys, (
    f"Invalid Bronze primary keys: {invalid_keys}"
)

### 4. Profile Bronze: NULL counts
Count NULL values in every data column of every Bronze table to decide which columns can be declared NOT NULL in Silver.

In [0]:
from pyspark.sql import functions as F

null_rows = []
for t in TABLES:
    df = spark.table(f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.{t}")
    data_cols = [c for c in df.columns if not c.startswith("_")]
    counts = df.agg(*[
        F.sum(F.col(c).isNull().cast("int")).alias(c) for c in data_cols
    ]).first().asDict()
    for c in data_cols:
        null_rows.append((t, c, int(counts[c] or 0)))

nulls_df = spark.createDataFrame(null_rows, "table_name string, column_name string, null_count long")

display(nulls_df.filter("null_count > 0"))
print(f"Columns checked: {len(null_rows)}, with NULL: {nulls_df.filter('null_count > 0').count()}")

### 5. Profile Bronze: numeric ranges
Measure minimum and maximum values of numeric columns to derive CHECK constraint bounds for Silver.

In [0]:
display(spark.sql(f"""
SELECT 'lineitem' AS table_name,
       min(l_quantity) AS q_min, max(l_quantity) AS q_max,
       min(l_extendedprice) AS ep_min, max(l_extendedprice) AS ep_max,
       min(l_discount) AS d_min, max(l_discount) AS d_max,
       min(l_tax) AS t_min, max(l_tax) AS t_max
FROM {TARGET_CATALOG}.{BRONZE_SCHEMA}.lineitem
"""))

display(spark.sql(f"""
SELECT min(p_retailprice) AS rp_min, max(p_retailprice) AS rp_max,
       min(p_size) AS size_min, max(p_size) AS size_max
FROM {TARGET_CATALOG}.{BRONZE_SCHEMA}.part
"""))

display(spark.sql(f"""
SELECT min(ps_supplycost) AS sc_min, max(ps_supplycost) AS sc_max,
       min(ps_availqty) AS qty_min, max(ps_availqty) AS qty_max
FROM {TARGET_CATALOG}.{BRONZE_SCHEMA}.partsupp
"""))

### 6. Profile Bronze: categorical values
List distinct values of categorical columns to build the allowed-value lists for Silver validation rules.

In [0]:
cats = [
    ("lineitem", "l_shipmode"),
    ("lineitem", "l_returnflag"),
    ("lineitem", "l_linestatus"),
    ("lineitem", "l_shipinstruct"),
    ("orders",   "o_orderstatus"),
    ("orders",   "o_orderpriority"),
    ("customer", "c_mktsegment"),
    ("part",     "p_mfgr"),
]

for t, c in cats:
    print(f"\n=== {t}.{c} ===")
    display(
        spark.table(f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.{t}")
        .groupBy(c).count().orderBy(c)
    )

In [0]:
display(
    spark.table(f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.part")
    .groupBy("p_brand").count().orderBy("p_brand")
)

### 7. Profile Bronze: referential integrity and consistency
Count orphan rows, brand-manufacturer mismatches and date inconsistencies in Bronze to see which rules will actually trigger.

In [0]:
from pyspark.sql import functions as F

def b(t): return spark.table(f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.{t}")

checks = []

def orphans(child, child_cols, parent, parent_cols, name):
    cond = [F.col(f"c.{x}") == F.col(f"p.{y}") for x, y in zip(child_cols, parent_cols)]
    cnt = b(child).alias("c").join(b(parent).alias("p"), cond[0] if len(cond) == 1 else cond[0] & cond[1], "left_anti").count()
    checks.append((name, cnt))

orphans("nation",   ["n_regionkey"],  "region",   ["r_regionkey"],  "nation -> region")
orphans("supplier", ["s_nationkey"],  "nation",   ["n_nationkey"],  "supplier -> nation")
orphans("customer", ["c_nationkey"],  "nation",   ["n_nationkey"],  "customer -> nation")
orphans("partsupp", ["ps_partkey"],   "part",     ["p_partkey"],    "partsupp -> part")
orphans("partsupp", ["ps_suppkey"],   "supplier", ["s_suppkey"],    "partsupp -> supplier")
orphans("orders",   ["o_custkey"],    "customer", ["c_custkey"],    "orders -> customer")
orphans("lineitem", ["l_orderkey"],   "orders",   ["o_orderkey"],   "lineitem -> orders")
orphans("lineitem", ["l_partkey", "l_suppkey"], "partsupp", ["ps_partkey", "ps_suppkey"], "lineitem -> partsupp (composite)")

# orders without line items
checks.append(("orders without lineitems",
               b("orders").alias("o").join(b("lineitem").alias("l"),
                   F.col("o.o_orderkey") == F.col("l.l_orderkey"), "left_anti").count()))

# brand vs manufacturer
checks.append(("brand/mfgr mismatch",
               b("part").filter(F.substring("p_brand", 7, 1) != F.substring("p_mfgr", 14, 1)).count()))

display(spark.createDataFrame(checks, "check_name string, violations long"))

In [0]:
li = b("lineitem").alias("l").join(b("orders").alias("o"), F.col("l.l_orderkey") == F.col("o.o_orderkey"))

display(li.agg(
    F.sum((F.col("l_shipdate") < F.col("o_orderdate")).cast("int")).alias("ship_before_order"),
    F.sum((F.col("l_receiptdate") < F.col("l_shipdate")).cast("int")).alias("receipt_before_ship"),
    F.sum((F.col("l_commitdate") < F.col("o_orderdate")).cast("int")).alias("commit_before_order"),
))

### 8. Create Silver tables: reference and dimension tables
Drop existing Silver tables in reverse dependency order, then create region, nation, supplier and customer with types, NOT NULL, PK and FK. PK and FK are informational in Unity Catalog; NOT NULL and CHECK are enforced. CHECK constraints are added in a later step.

In [0]:
S = f"{TARGET_CATALOG}.{SILVER_SCHEMA}"
drop_order = [
    "lineitem",
    "orders",
    "partsupp",
    "part",
    "brand",
    "customer",
    "supplier",
    "nation",
    "region",
    "quarantine",
]

for table in drop_order:
    spark.sql(f"DROP TABLE IF EXISTS {S}.{table}")

print("Previous Silver tables removed.")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {S}.region (
  r_regionkey BIGINT NOT NULL,
  r_name      STRING NOT NULL,
  r_comment   STRING,
  CONSTRAINT pk_region PRIMARY KEY (r_regionkey)
)""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {S}.nation (
  n_nationkey BIGINT NOT NULL,
  n_name      STRING NOT NULL,
  n_regionkey BIGINT NOT NULL,
  n_comment   STRING,
  CONSTRAINT pk_nation PRIMARY KEY (n_nationkey),
  CONSTRAINT fk_nation_region FOREIGN KEY (n_regionkey) REFERENCES {S}.region(r_regionkey)
)""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {S}.supplier (
  s_suppkey   BIGINT        NOT NULL,
  s_name      STRING        NOT NULL,
  s_address   STRING        NOT NULL,
  s_nationkey BIGINT        NOT NULL,
  s_phone     STRING        NOT NULL,
  s_acctbal   DECIMAL(18,2) NOT NULL,
  s_comment   STRING,
  CONSTRAINT pk_supplier PRIMARY KEY (s_suppkey),
  CONSTRAINT fk_supplier_nation FOREIGN KEY (s_nationkey) REFERENCES {S}.nation(n_nationkey)
)""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {S}.customer (
  c_custkey    BIGINT        NOT NULL,
  c_name       STRING        NOT NULL,
  c_address    STRING        NOT NULL,
  c_nationkey  BIGINT        NOT NULL,
  c_phone      STRING        NOT NULL,
  c_acctbal    DECIMAL(18,2) NOT NULL,
  c_mktsegment STRING        NOT NULL,
  c_comment    STRING,
  CONSTRAINT pk_customer PRIMARY KEY (c_custkey),
  CONSTRAINT fk_customer_nation FOREIGN KEY (c_nationkey) REFERENCES {S}.nation(n_nationkey)
)""")

### 9. Create Silver tables: part, partsupp, orders and lineitem
Create the remaining Silver tables with types, NOT NULL, PK and FK. The composite foreign key from lineitem to partsupp is informational; its integrity is enforced by the load rules in a later step.

In [0]:
spark.sql(f"""
CREATE TABLE {S}.brand (
    p_brand STRING NOT NULL,
    p_mfgr  STRING NOT NULL,
    CONSTRAINT pk_brand PRIMARY KEY (p_brand)
)
""")

spark.sql(f"""
CREATE TABLE {S}.part (
    p_partkey     BIGINT        NOT NULL,
    p_name        STRING        NOT NULL,
    p_brand       STRING        NOT NULL,
    p_type        STRING        NOT NULL,
    p_size        INT           NOT NULL,
    p_container   STRING        NOT NULL,
    p_retailprice DECIMAL(18,2) NOT NULL,
    p_comment     STRING,

    CONSTRAINT pk_part PRIMARY KEY (p_partkey),
    CONSTRAINT fk_part_brand FOREIGN KEY (p_brand)
        REFERENCES {S}.brand(p_brand)
)
""")

spark.sql(f"""
CREATE TABLE {S}.partsupp (
  ps_partkey    BIGINT        NOT NULL,
  ps_suppkey    BIGINT        NOT NULL,
  ps_availqty   INT           NOT NULL,
  ps_supplycost DECIMAL(18,2) NOT NULL,
  ps_comment    STRING,
  CONSTRAINT pk_partsupp PRIMARY KEY (ps_partkey, ps_suppkey),
  CONSTRAINT fk_ps_part FOREIGN KEY (ps_partkey) REFERENCES {S}.part(p_partkey),
  CONSTRAINT fk_ps_supplier FOREIGN KEY (ps_suppkey) REFERENCES {S}.supplier(s_suppkey)
)""")

spark.sql(f"""
CREATE TABLE {S}.orders (
  o_orderkey      BIGINT        NOT NULL,
  o_custkey       BIGINT        NOT NULL,
  o_orderstatus   STRING        NOT NULL,
  o_totalprice    DECIMAL(18,2) NOT NULL,
  o_orderdate     DATE          NOT NULL,
  o_orderpriority STRING        NOT NULL,
  o_clerk         STRING        NOT NULL,
  o_shippriority  INT           NOT NULL,
  o_comment       STRING,
  CONSTRAINT pk_orders PRIMARY KEY (o_orderkey),
  CONSTRAINT fk_orders_customer FOREIGN KEY (o_custkey) REFERENCES {S}.customer(c_custkey)
)""")

spark.sql(f"""
CREATE TABLE {S}.lineitem (
  l_orderkey      BIGINT        NOT NULL,
  l_partkey       BIGINT        NOT NULL,
  l_suppkey       BIGINT        NOT NULL,
  l_linenumber    INT           NOT NULL,
  l_quantity      DECIMAL(18,2) NOT NULL,
  l_extendedprice DECIMAL(18,2) NOT NULL,
  l_discount      DECIMAL(18,2) NOT NULL,
  l_tax           DECIMAL(18,2) NOT NULL,
  l_returnflag    STRING        NOT NULL,
  l_linestatus    STRING        NOT NULL,
  l_shipdate      DATE          NOT NULL,
  l_commitdate    DATE          NOT NULL,
  l_receiptdate   DATE          NOT NULL,
  l_shipinstruct  STRING        NOT NULL,
  l_shipmode      STRING        NOT NULL,
  l_comment       STRING,
  CONSTRAINT pk_lineitem PRIMARY KEY (l_orderkey, l_linenumber),
  CONSTRAINT fk_li_orders FOREIGN KEY (l_orderkey) REFERENCES {S}.orders(o_orderkey),
  CONSTRAINT fk_li_partsupp FOREIGN KEY (l_partkey, l_suppkey)
    REFERENCES {S}.partsupp(ps_partkey, ps_suppkey)
)""")

### 10. Add CHECK constraints to Silver tables
CHECK constraints cannot be declared inside CREATE TABLE in Databricks, so they are added with ALTER TABLE. The bounds and allowed-value lists come from the Bronze profiling steps. CHECK is enforced on every write.

In [0]:
checks = [
    ("customer", "ck_customer_segment",
     "c_mktsegment IN ('AUTOMOBILE','BUILDING','FURNITURE','HOUSEHOLD','MACHINERY')"),

    ("brand", "ck_brand_mfgr",
    "p_mfgr RLIKE '^Manufacturer#[1-5]$'"),

    ("brand", "ck_brand_name",
    "p_brand RLIKE '^Brand#[1-5][1-5]$'"),

    ("brand", "ck_brand_mfgr_match",
    "substring(p_brand, 7, 1) = substring(p_mfgr, 14, 1)"),
    ("part", "ck_part_size", "p_size BETWEEN 1 AND 50"),
    ("part", "ck_part_price", "p_retailprice > 0"),

    ("partsupp", "ck_ps_qty", "ps_availqty >= 0"),
    ("partsupp", "ck_ps_cost", "ps_supplycost > 0"),

    ("orders", "ck_orders_status", "o_orderstatus IN ('F','O','P')"),
    ("orders", "ck_orders_priority",
     "o_orderpriority IN ('1-URGENT','2-HIGH','3-MEDIUM','4-NOT SPECIFIED','5-LOW')"),
    ("orders", "ck_orders_price", "o_totalprice > 0"),

    ("lineitem", "ck_li_quantity", "l_quantity > 0"),
    ("lineitem", "ck_li_price", "l_extendedprice > 0"),
    ("lineitem", "ck_li_discount", "l_discount BETWEEN 0 AND 0.10"),
    ("lineitem", "ck_li_tax", "l_tax BETWEEN 0 AND 0.08"),
    ("lineitem", "ck_li_returnflag", "l_returnflag IN ('A','N','R')"),
    ("lineitem", "ck_li_linestatus", "l_linestatus IN ('F','O')"),
    ("lineitem", "ck_li_shipinstruct",
     "l_shipinstruct IN ('COLLECT COD','DELIVER IN PERSON','NONE','TAKE BACK RETURN')"),
    ("lineitem", "ck_li_shipmode",
     "l_shipmode IN ('AIR','FOB','MAIL','RAIL','REG AIR','SHIP','TRUCK')"),
    ("lineitem", "ck_li_ship_receipt", "l_receiptdate >= l_shipdate"),
]

for table, name, expr in checks:
    spark.sql(f"ALTER TABLE {S}.{table} DROP CONSTRAINT IF EXISTS {name}")
    spark.sql(f"ALTER TABLE {S}.{table} ADD CONSTRAINT {name} CHECK ({expr})")

print(f"CHECK constraints added: {len(checks)}")

### 11. Create the quarantine table
Create a single quarantine table that stores every rejected row as JSON together with the names of the rules it violated. It is recreated on each run so that reruns never duplicate rows.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {S}.quarantine (
  source_table   STRING    NOT NULL,
  failed_rules   STRING    NOT NULL,
  row_json       STRING    NOT NULL,
  _run_id        STRING,
  quarantined_at TIMESTAMP NOT NULL
)""")

### 12. Define validation helpers
`split_good_bad` evaluates a dictionary of rules and splits a DataFrame into valid rows and rejected rows, recording every violated rule. A NULL rule result counts as a violation. `to_quarantine` converts rejected rows into quarantine records, one record per rejected row.

In [0]:
from pyspark.sql import functions as F

META_COLS = ["_loaded_at", "_source_table", "_run_id"]

def split_good_bad(df, rules):
    """rules: {rule_name: condition that must be TRUE for a valid row}"""
    flags = F.array_compact(F.array(*[
        F.when(~F.coalesce(cond, F.lit(False)), F.lit(name))
        for name, cond in rules.items()
    ]))
    df = df.withColumn("_failed", flags)
    good = df.filter(F.size("_failed") == 0).drop("_failed")
    bad = df.filter(F.size("_failed") > 0)
    return good, bad

def to_quarantine(bad, table_name):
    src_cols = [c for c in bad.columns if c not in META_COLS + ["_failed"]]
    return bad.select(
        F.lit(table_name).alias("source_table"),
        F.concat_ws(",", "_failed").alias("failed_rules"),
        F.to_json(F.struct(*src_cols)).alias("row_json"),
        F.col("_run_id"),
        F.current_timestamp().alias("quarantined_at"),
    )

In [0]:
display(spark.sql(f"DESCRIBE TABLE {S}.quarantine"))

### 13. Load Silver: region, nation, supplier and customer
`load_silver` applies the rules, writes valid rows to Silver, writes rejected rows to quarantine and reconciles the counts (Bronze = Silver + quarantine). Child tables are checked against the already loaded Silver parents, so rows whose parent was rejected are quarantined as orphans.

In [0]:
def bronze(t):
    return spark.table(f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.{t}")

def parent_keys(table, key, alias):
    """Keys of the already loaded (clean) Silver parent table."""
    return spark.table(f"{S}.{table}").select(F.col(key).alias(alias))

def load_silver(table, df, rules, helpers=()):
    good, bad = split_good_bad(df, rules)
    bad = bad.drop(*helpers)

    cols = spark.table(f"{S}.{table}").columns   # insertInto writes by position
    good.select(*cols).write.mode("overwrite").insertInto(f"{S}.{table}")

    # makes the cell safe to rerun on its own
    spark.sql(f"DELETE FROM {S}.quarantine WHERE source_table = '{table}'")
    to_quarantine(bad, table).write.mode("append").saveAsTable(f"{S}.quarantine")

    n_bronze = bronze(table).count()
    n_silver = spark.table(f"{S}.{table}").count()
    n_quar = spark.table(f"{S}.quarantine").filter(F.col("source_table") == table).count()
    assert n_bronze == n_silver + n_quar, (
    f"Row reconciliation failed for {table}: "
    f"Bronze={n_bronze}, Silver={n_silver}, quarantine={n_quar}"
    )
    return (table, n_bronze, n_silver, n_quar, n_bronze == n_silver + n_quar)

def not_blank(c):
    return F.trim(F.col(c)) != ""

load_results = []

In [0]:
# region (no parent)
load_results.append(load_silver("region", bronze("region"), {
    "region_name_blank": not_blank("r_name"),
}))

# nation -> region
df = bronze("nation").join(parent_keys("region", "r_regionkey", "_p_region"),
                           F.col("n_regionkey") == F.col("_p_region"), "left")
load_results.append(load_silver("nation", df, {
    "nation_name_blank": not_blank("n_name"),
    "orphan_region":     F.col("_p_region").isNotNull(),
}, helpers=["_p_region"]))

# supplier -> nation
df = bronze("supplier").join(parent_keys("nation", "n_nationkey", "_p_nation"),
                             F.col("s_nationkey") == F.col("_p_nation"), "left")
load_results.append(load_silver("supplier", df, {
    "supplier_name_blank":    not_blank("s_name"),
    "supplier_address_blank": not_blank("s_address"),
    "supplier_phone_blank":   not_blank("s_phone"),
    "orphan_nation":          F.col("_p_nation").isNotNull(),
}, helpers=["_p_nation"]))

# customer -> nation
df = bronze("customer").join(parent_keys("nation", "n_nationkey", "_p_nation"),
                             F.col("c_nationkey") == F.col("_p_nation"), "left")
load_results.append(load_silver("customer", df, {
    "customer_name_blank": not_blank("c_name"),
    "invalid_segment":     F.col("c_mktsegment").isin(
        "AUTOMOBILE", "BUILDING", "FURNITURE", "HOUSEHOLD", "MACHINERY"),
    "orphan_nation":       F.col("_p_nation").isNotNull(),
}, helpers=["_p_nation"]))

display(spark.createDataFrame(
    load_results,
    "table_name string, bronze_rows long, silver_rows long, quarantine_rows long, reconciled boolean",
))

### 14. Load Silver: part and partsupp
Load part with the brand and manufacturer rules (non-blank, valid format, brand prefix equals manufacturer number). Load partsupp against the clean part and supplier tables and require a positive supply cost.

In [0]:
brand_source = bronze("part")

valid_brand_pair = (
    F.col("p_brand").isNotNull()
    & F.col("p_mfgr").isNotNull()
    & F.col("p_brand").rlike(r"^Brand#[1-5][1-5]$")
    & F.col("p_mfgr").rlike(r"^Manufacturer#[1-5]$")
    & (
        F.substring("p_brand", 7, 1)
        == F.substring("p_mfgr", 14, 1)
    )
)

brand_df = (
    brand_source
    .filter(valid_brand_pair)
    .select("p_brand", "p_mfgr")
    .distinct()
)

ambiguous_brands = (
    brand_df
    .groupBy("p_brand")
    .agg(F.countDistinct("p_mfgr").alias("manufacturers"))
    .filter(F.col("manufacturers") != 1)
    .count()
)

assert ambiguous_brands == 0, (
    "A brand is associated with multiple manufacturers"
)

assert brand_df.count() > 0, "No valid brands found"

(
    brand_df
    .select("p_brand", "p_mfgr")
    .write.mode("overwrite")
    .insertInto(f"{S}.brand")
)

display(spark.table(f"{S}.brand"))

In [0]:
# part -> brand
df = (
    bronze("part")
    .join(
        parent_keys("brand", "p_brand", "_p_brand"),
        F.col("p_brand") == F.col("_p_brand"),
        "left",
    )
)

load_results.append(load_silver(
    "part",
    df,
    {
        "brand_blank": not_blank("p_brand"),
        "mfgr_blank": not_blank("p_mfgr"),

        "brand_format":
            F.col("p_brand").rlike(r"^Brand#[1-5][1-5]$"),

        "mfgr_format":
            F.col("p_mfgr").rlike(r"^Manufacturer#[1-5]$"),

        "brand_mfgr_mismatch":
            F.substring("p_brand", 7, 1)
            == F.substring("p_mfgr", 14, 1),

        "orphan_brand": F.col("_p_brand").isNotNull(),

        "retailprice_positive": F.col("p_retailprice") > 0,
        "size_range": F.col("p_size").between(1, 50),
    },
    helpers=["_p_brand"],
))

# partsupp -> part, supplier
df = (bronze("partsupp")
      .join(parent_keys("part", "p_partkey", "_p_part"),
            F.col("ps_partkey") == F.col("_p_part"), "left")
      .join(parent_keys("supplier", "s_suppkey", "_p_supp"),
            F.col("ps_suppkey") == F.col("_p_supp"), "left"))
load_results.append(load_silver("partsupp", df, {
    "orphan_part":         F.col("_p_part").isNotNull(),
    "orphan_supplier":     F.col("_p_supp").isNotNull(),
    "supplycost_positive": F.col("ps_supplycost") > 0,
    "availqty_nonnegative": F.col("ps_availqty") >= 0,
}, helpers=["_p_part", "_p_supp"]))

display(spark.createDataFrame(
    load_results[-2:],
    "table_name string, bronze_rows long, silver_rows long, quarantine_rows long, reconciled boolean",
))

### 15. Load Silver: orders and lineitem
Load orders against the clean customer table. Load lineitem against the clean orders and partsupp tables: the (part, supplier) pair is matched with a composite join, and shipment dates are validated against each other and against the parent order date.

In [0]:
# orders -> customer
df = bronze("orders").join(parent_keys("customer", "c_custkey", "_p_cust"),
                           F.col("o_custkey") == F.col("_p_cust"), "left")
load_results.append(load_silver("orders", df, {
    "orphan_customer":  F.col("_p_cust").isNotNull(),
    "invalid_status":   F.col("o_orderstatus").isin("F", "O", "P"),
    "invalid_priority": F.col("o_orderpriority").isin(
        "1-URGENT", "2-HIGH", "3-MEDIUM", "4-NOT SPECIFIED", "5-LOW"),
    "totalprice_positive": F.col("o_totalprice") > 0,
}, helpers=["_p_cust"]))

display(spark.createDataFrame(
    load_results[-1:],
    "table_name string, bronze_rows long, silver_rows long, quarantine_rows long, reconciled boolean",
))

In [0]:
def lineitem_rules():
    return {
        "orphan_order":         F.col("_p_order").isNotNull(),
        "orphan_partsupp":      F.col("_p_ps_part").isNotNull(),
        "ship_before_order":    F.col("_o_orderdate").isNull() | (F.col("l_shipdate") >= F.col("_o_orderdate")),
        "receipt_before_ship":  F.col("l_receiptdate") >= F.col("l_shipdate"),
        "quantity_positive":    F.col("l_quantity") > 0,
        "price_positive":       F.col("l_extendedprice") > 0,
        "discount_range":       F.col("l_discount").between(0, 0.10),
        "tax_range":            F.col("l_tax").between(0, 0.08),
        "invalid_returnflag":   F.col("l_returnflag").isin("A", "N", "R"),
        "invalid_linestatus":   F.col("l_linestatus").isin("F", "O"),
        "invalid_shipinstruct": F.col("l_shipinstruct").isin(
            "COLLECT COD", "DELIVER IN PERSON", "NONE", "TAKE BACK RETURN"),
        "invalid_shipmode":     F.col("l_shipmode").isin(
            "AIR", "FOB", "MAIL", "RAIL", "REG AIR", "SHIP", "TRUCK"),
    }

### 16. Load Silver: lineitem
Load lineitem against the clean orders and partsupp tables. The (part, supplier) pair is matched with a composite join, and shipment dates are validated against each other and against the parent order date. Only the key and date columns of the parents are joined to keep the shuffle small.

In [0]:
orders_ref = (spark.table(f"{S}.orders")
              .select(F.col("o_orderkey").alias("_p_order"),
                      F.col("o_orderdate").alias("_o_orderdate")))

ps_ref = (spark.table(f"{S}.partsupp")
          .select(F.col("ps_partkey").alias("_p_ps_part"),
                  F.col("ps_suppkey").alias("_p_ps_supp")))

df = (bronze("lineitem")
      .join(orders_ref, F.col("l_orderkey") == F.col("_p_order"), "left")
      .join(ps_ref,
            (F.col("l_partkey") == F.col("_p_ps_part")) &
            (F.col("l_suppkey") == F.col("_p_ps_supp")),
            "left"))

load_results.append(load_silver(
    "lineitem", df, lineitem_rules(),
    helpers=["_p_order", "_o_orderdate", "_p_ps_part", "_p_ps_supp"]))

display(spark.createDataFrame(
    load_results[-1:],
    "table_name string, bronze_rows long, silver_rows long, quarantine_rows long, reconciled boolean",
))

### 17. Reconcile monetary totals between Bronze and Silver
Compare the same monetary metrics in Bronze and Silver. Because rejected rows are kept in quarantine, Bronze must equal Silver plus the quarantined amount. The run fails if any difference remains.

In [0]:
metrics = [
    ("orders",   "total_order_price",    "o_totalprice"),
    ("lineitem", "total_extended_price", "l_extendedprice"),
    ("lineitem", "net_revenue",          "l_extendedprice * (1 - l_discount)"),
]

def total(table_name, expression):
    return (spark.table(table_name)
            .agg(F.sum(F.expr(expression).cast("decimal(28,6)")).alias("t"))
            .first()["t"]) or 0

def quarantined_total(table, expression):
    q = (spark.table(f"{S}.quarantine")
         .filter(F.col("source_table") == table)
         .select(F.from_json("row_json", spark.table(f"{S}.{table}").schema).alias("r"))
         .select("r.*"))
    return (q.agg(F.sum(F.expr(expression).cast("decimal(28,6)")).alias("t"))
             .first()["t"]) or 0

money_rows = []
for table, metric, expr in metrics:
    b_total = total(f"{TARGET_CATALOG}.{BRONZE_SCHEMA}.{table}", expr)
    s_total = total(f"{S}.{table}", expr)
    q_total = quarantined_total(table, expr)
    diff = b_total - (s_total + q_total)
    money_rows.append((metric, str(b_total), str(s_total), str(q_total), str(diff), diff == 0))

display(spark.createDataFrame(
    money_rows,
    "metric string, bronze_total string, silver_total string, quarantine_total string, difference string, totals_match boolean",
))

assert all(r[5] for r in money_rows), "Bronze != Silver + quarantine for some monetary metric"

### 18. Demo: validation on deliberately corrupted rows
Take a small sample of real Bronze parts, append hand-made bad rows and run them through the same `split_good_bad` and `to_quarantine` functions used by the pipeline. Nothing is written to the real Silver tables.

In [0]:
from pyspark.sql import Row
from decimal import Decimal

sample = bronze("part").limit(20)

bad_rows = spark.createDataFrame([
    Row(p_partkey=9000001, p_name="blank brand",         p_mfgr="Manufacturer#3", p_brand="",         p_type="X", p_size=10, p_container="BOX", p_retailprice=Decimal("100.00"), p_comment=None),
    Row(p_partkey=9000002, p_name="brand/mfgr mismatch", p_mfgr="Manufacturer#3", p_brand="Brand#12", p_type="X", p_size=10, p_container="BOX", p_retailprice=Decimal("100.00"), p_comment=None),
    Row(p_partkey=9000003, p_name="null brand",          p_mfgr="Manufacturer#2", p_brand=None,       p_type="X", p_size=10, p_container="BOX", p_retailprice=Decimal("100.00"), p_comment=None),
    Row(p_partkey=9000004, p_name="negative price",      p_mfgr="Manufacturer#1", p_brand="Brand#11", p_type="X", p_size=10, p_container="BOX", p_retailprice=Decimal("-5.00"),  p_comment=None),
    Row(p_partkey=9000005, p_name="valid control",       p_mfgr="Manufacturer#3", p_brand="Brand#32", p_type="X", p_size=10, p_container="BOX", p_retailprice=Decimal("100.00"), p_comment=None),
], schema="p_partkey long, p_name string, p_mfgr string, p_brand string, p_type string, p_size int, p_container string, p_retailprice decimal(18,2), p_comment string")

# add the same metadata columns that Bronze rows have
bad_rows = (bad_rows
            .withColumn("_loaded_at", F.current_timestamp())
            .withColumn("_source_table", F.lit("demo"))
            .withColumn("_run_id", F.lit("demo-run")))

demo_df = sample.unionByName(bad_rows)

demo_rules = {
    "brand_blank":         not_blank("p_brand"),
    "mfgr_blank":          not_blank("p_mfgr"),
    "brand_format":        F.col("p_brand").rlike(r"^Brand#[1-5][1-5]$"),
    "mfgr_format":         F.col("p_mfgr").rlike(r"^Manufacturer#[1-5]$"),
    "brand_mfgr_mismatch": F.substring("p_brand", 7, 1) == F.substring("p_mfgr", 14, 1),
    "retailprice_positive": F.col("p_retailprice") > 0,
    "size_range":          F.col("p_size").between(1, 50),
}

good, bad = split_good_bad(demo_df, demo_rules)
print(f"input={demo_df.count()}, good={good.count()}, quarantined={bad.count()}")
display(to_quarantine(bad, "part").select("source_table", "failed_rules", "row_json"))

### 19. Demo: lineitem validation on deliberately corrupted rows
Take one real Bronze line item, create copies with one defect each, and run them through the same joins and `lineitem_rules()` used by the pipeline. Nothing is written to the real Silver tables.

In [0]:
from functools import reduce
from decimal import Decimal

base = bronze("lineitem").limit(1)

variants = [
    ("control",             base),
    ("receipt_before_ship", base.withColumn("l_receiptdate", F.date_sub("l_shipdate", 1))),
    ("ship_before_order",   base.withColumn("l_shipdate", F.lit("1900-01-01").cast("date"))),
    ("orphan_partsupp",     base.withColumn("l_partkey", F.lit(-1).cast("long"))),
    ("invalid_shipmode",    base.withColumn("l_shipmode", F.lit("PIGEON"))),
    ("discount_range",      base.withColumn("l_discount", F.lit(Decimal("0.50")).cast("decimal(18,2)"))),
    ("orphan_order",        base.withColumn("l_orderkey", F.lit(-1).cast("long"))),
]
demo_li = reduce(lambda a, b: a.unionByName(b),
                 [df.withColumn("_scenario", F.lit(name)) for name, df in variants])

demo_df = (demo_li
           .join(orders_ref, F.col("l_orderkey") == F.col("_p_order"), "left")
           .join(ps_ref, (F.col("l_partkey") == F.col("_p_ps_part")) &
                         (F.col("l_suppkey") == F.col("_p_ps_supp")), "left"))

good, bad = split_good_bad(demo_df, lineitem_rules())
print(f"input={demo_df.count()}, good={good.count()}, quarantined={bad.count()}")

display(bad.select("_scenario", F.concat_ws(",", "_failed").alias("failed_rules")))